## tl;dr

마법사만 MP 보너스를 최대 +15%p, 기본 스킬 확률 상한을 35%로 올리는 안은 직업 개성을 강화한다. 90→100레벨 시간당 처치 +3.51%, 시간당 스킬 사용 +10.41%, 8시간 간격에서 100레벨까지 성장속도 +1.72%였다. 앱 적용은 하지 않았다.


## Context & Methods

### Key Assumptions

성직자는 WIS/CHA. HP 8→10시간, 탐색 최소 4초, 판매 보너스 최대 20%를 유지한다. 마법사 seed 6~13을 재계산하고 나머지 5개 직업의 같은 seed 결과는 재사용한다. 총 스킬 비율, 시간당 시전, 시간당 처치, 100레벨 도달 속도는 다른 지표다. 접속 간격은 저장시간 기반 근사이며 실제 사용자 로그가 아니다.

재현: run_mage35_stat_probe.py, review_mage35.py. 네트워크와 DB 파일을 차단한 샌드박스에서 실행했다. 일반 Python으로 코드 셀을 순차 실행했으며 Jupyter 커널 자체 실행은 미검증이다. nbconvert/ipykernel이 설치된 환경에서는 `python -m jupyter nbconvert --execute --to notebook --inplace mage35-review.ipynb`로 검증할 수 있다.


## Data

### 1. Load local evidence and recheck calculations


In [1]:
from pathlib import Path
import sys,json,numpy as np,pandas as pd
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tools/analysis/review_mage35.py').exists())
sys.path.insert(0,str(root/'tools/analysis'))
from review_mage35 import review
comparison,summary=review()
print('Mage paths:',len(summary['input_seeds']))
assert summary['app_source_unchanged'] and summary['other_class_paths_unchanged'] and summary['calendar_spot_check_passed']


Mage paths: 8


## Results

### 2. Compare actual late-game counters


In [2]:
late=pd.DataFrame(summary['late_game']).set_index('variant')
print(late[['pooled_cast_pct','pooled_kills_per_hour','pooled_casts_per_hour']].round(4).to_string())
for field in ['pooled_kills_per_hour','pooled_casts_per_hour']:
    print(field,'gain_pct',100*(late.loc['mage35',field]/late.loc['mage30',field]-1))
assert np.isclose(late.loc['mage35','pooled_kills_per_hour']/late.loc['mage30','pooled_kills_per_hour']-1,.03514340734526189)


         pooled_cast_pct  pooled_kills_per_hour  pooled_casts_per_hour
variant                                                               
mage30           29.8364               192.0662               316.4215
mage35           34.6774               198.8161               349.3491
pooled_kills_per_hour gain_pct 3.514340734526189
pooled_casts_per_hour gain_pct 10.406219135617345


### 3. Separate fixed probability from whole-growth efficiency


In [3]:
print(pd.DataFrame(summary['theoretical_fixed_proc']).round(4).to_string(index=False))
print(comparison[(comparison.level==100)&(comparison.gap_h.isin([8,12]))&(comparison.session_min==12)][['variant','gap_h','hero_class','speed_vs_paladin_pct']].round(4).to_string(index=False))
print(pd.DataFrame(summary['comparisons']).round(4).to_string(index=False))
assert np.isclose(summary['theoretical_fixed_proc'][1]['effective_pct'],35.035573199950406)


 raw_pct  effective_pct
      30        30.1000
      35        35.0356
variant  gap_h hero_class  speed_vs_paladin_pct
 mage30    8.0    WARRIOR                0.0139
 mage30    8.0      ROGUE                2.3709
 mage30    8.0     RANGER                2.9726
 mage30    8.0       MAGE                2.1703
 mage30    8.0     CLERIC                1.2159
 mage30    8.0    PALADIN                0.0000
 mage30   12.0    WARRIOR                5.5062
 mage30   12.0      ROGUE                2.4344
 mage30   12.0     RANGER                3.1365
 mage30   12.0       MAGE                2.3683
 mage30   12.0     CLERIC                1.4473
 mage30   12.0    PALADIN                0.0000
 mage35    8.0    WARRIOR                0.0139
 mage35    8.0      ROGUE                2.3709
 mage35    8.0     RANGER                2.9726
 mage35    8.0       MAGE                3.9284
 mage35    8.0     CLERIC                1.2159
 mage35    8.0    PALADIN                0.0000
 mage35   12.0  

## Takeaways

35%는 전투·연출의 개성을 강화하지만 전체 성장속도를 16.7% 높이는 것은 아니다. 8시간 간격에서는 마법사가 레인저보다 약 0.93% 빠르고, 12시간 간격에서는 전사가 여전히 가장 빠르다. 초기 성장의 차이는 작고 후반에 효과가 커진다. 판매·가방의 경제적 가치는 성장속도와 동일 점수로 합산하지 않았다. 8개 seed의 결과라 전체 난수, 실제 접속 행동, 장기 경제를 보장하지 않는다.
